# 5-Class Emergency Department Triage Classifier: Base LightGBM vs. Conditional ESI 4 & 5 Logistic Calibrator (`models/train_fedmm_classifier.ipynb`)

This notebook trains an **Optuna-tuned 5-Class LightGBM Classifier** and implements a **Conditional Stage-2 Logistic Calibrator** specifically targeting low-acuity classes (**ESI 4 & ESI 5**) on the **Federated Multi-Modal Emergency Dataset (`datasets/fedmml_ed_triage_dataset.csv`)**:

### 🔬 Two Comparative Pipelines
1. **Model 1: Base Optuna-Tuned LightGBM**:
   - Primary multiclass classifier trained with balanced weights and tuned on the validation set.
2. **Model 2: LightGBM + Conditional ESI 4/5 Logistic Calibrator (Approach 2)**:
   - **Stage 1**: LightGBM outputs predicted probability vector $\mathbf{P} = [P_1, P_2, P_3, P_4, P_5]$.
   - **Stage 2 (Specialist Calibrator)**: High-acuity probabilities ($P_1, P_2, P_3$) remain completely untouched. For low-acuity mass $M_{45} = P_4 + P_5$, a dedicated binary `LogisticRegression(class_weight='balanced')` fit on validation samples of ESI 4 & 5 predicts conditional probability $q_5 = P(Y = 5 \mid Y \in \{4, 5\})$ and $q_4 = 1 - q_5$:
     $$\widetilde{P}_4 = (P_4 + P_5) \cdot q_4, \quad \widetilde{P}_5 = (P_4 + P_5) \cdot q_5, \quad \widetilde{P}_{1..3} = P_{1..3}$$

### 📊 Evaluation & Confidence Intervals
- Evaluates both models on the held-out test set with **DeLong 95% Confidence Intervals for AUROC**.
- Compares performance deltas ($\Delta = \text{Calibrated} - \text{Base}$) across all classes and macro-averages.

```mermaid
flowchart TD
    RawData["FedMML Dataset (87,234 encounters)"] --> Split["Stratified 3-Way Split via config/triage_conf.json (Train / Val / Test)"]
    Split --> EDA["Plot Feature & Target Class Distributions"]
    Split --> MICE["MICE Imputation (fit on Train, transform Val & Test)"]
    MICE --> Scale["StandardScaler Normalization (fit on Train, transform Val & Test)"]
    
    Scale --> OptunaLGBM["Optuna Study (TPESampler): Tune LightGBM on Validation Set"]
    OptunaLGBM --> BestLGBM["Train Base Tuned LightGBM Model"]
    
    BestLGBM --> CalibStage["Train Stage-2 Conditional ESI 4/5 Logistic Calibrator on Validation Set"]
    
    BestLGBM & CalibStage --> EvalTest["Holdout Test Benchmark with DeLong 95% AUROC CIs"]
    EvalTest --> CompTable["Direct Comparative Analysis: Base LightGBM vs Calibrated LightGBM"]
    EvalTest --> DiagPlots["Comparative Visualizations: 1x2 Confusion Matrices, 1x2 ROC Curves with 95% CIs, Feature Importance"]
```

In [ ]:
# ---------------------------------------------------------
# Step 1: Load FedMML Dataset, Encode 'sex', & Stratified 3-Way Split via triage_conf.json
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import optuna
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

# 1. Load Partitioning Configuration
config_path = f"{ROOT}/config/triage_conf.json"
with open(config_path, 'r') as f:
    config = json.load(f)

test_size = config['training']['test_size']
val_size  = config['training']['val_size']
seed_val  = config['training']['random_state']

print(f"Loaded Configuration from {config_path}:")
print(f"  * Test Size Fraction       = {test_size:.2f} ({test_size*100:.1f}%)")
print(f"  * Validation Size Fraction = {val_size:.2f} ({val_size*100:.1f}%)")
print(f"  * Random State Seed        = {seed_val}")
print("-" * 85)

# 2. Load Raw FedMML Dataset
data_path = f"{ROOT}/datasets/fedmml_ed_triage_dataset.csv"
print(f"Loading FedMML dataset from: {data_path}...")
df = pd.read_csv(data_path)
total_encounters = len(df)

required_features = ['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']
target_col = 'esi_level'

# 3. Encode 'sex' Feature (M -> 1, F -> 0)
df['sex_encoded'] = df['sex'].astype(str).str.strip().str.upper().map({'M': 1.0, 'MALE': 1.0, 'F': 0.0, 'FEMALE': 0.0})
feature_names = ['age', 'sex_encoded', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']

df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)

print("=" * 85)
print(f"  FEDMML DATASET: {total_encounters:,} Total Encounters")
print("=" * 85)
print("Missing Value Count per Feature (to be imputed using MICE):")
print(df[feature_names].isnull().sum())
print("-" * 85)
print("Target ESI Distribution:")
esi_dist = df[target_col].value_counts().sort_index()
for cls_val, cnt in esi_dist.items():
    print(f"  * ESI Level {cls_val} : {cnt:,} encounters ({cnt/len(df)*100:.2f}%)")
print("=" * 85)

# 4. Stratified 3-Way Partitioning (Referencing triage_conf.json)
X_all = df[feature_names].values
y_all = df[target_col].values

# (a) Extract Stratified Holdout Test Set
X_rem_raw, X_test_raw, y_rem, y_test = train_test_split(
    X_all, y_all, test_size=test_size, stratify=y_all, random_state=seed_val
)

# (b) Extract Stratified Validation Set from remainder
val_adj_fraction = val_size / (1.0 - test_size)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_rem_raw, y_rem, test_size=val_adj_fraction, stratify=y_rem, random_state=seed_val + 1
)

print(f"Stratified Partition Complete (Referenced from triage_conf.json):")
print(f"  * Train Set      : {len(X_train_raw):,} encounters ({len(X_train_raw)/len(df)*100:.1f}%)")
print(f"  * Validation Set : {len(X_val_raw):,} encounters ({len(X_val_raw)/len(df)*100:.1f}%)")
print(f"  * Holdout Test   : {len(X_test_raw):,} encounters ({len(X_test_raw)/len(df)*100:.1f}%)")

In [ ]:
# ---------------------------------------------------------
# Step 2: Exploratory Data Analysis — Feature & Target Class Distributions
# ---------------------------------------------------------
plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

# 1. Plot Target Class Distributions (Overall Dataset & Stratified Partitions)
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

# Left: Overall Dataset Target Distribution
esi_counts = df[target_col].value_counts().sort_index()
esi_pcts   = df[target_col].value_counts(normalize=True).sort_index() * 100

bars = axes[0].bar(esi_labels, esi_counts.values, color=esi_colors, edgecolor='black', alpha=0.85)
axes[0].set_title(f"Overall FedMML Target ESI Class Distribution (N = {len(df):,})", fontsize=12, fontweight='bold', pad=10)
axes[0].set_xlabel("ESI Acuity Level", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Number of Encounters", fontsize=11, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.3, axis='y')

for bar, count, pct in zip(bars, esi_counts.values, esi_pcts.values):
    axes[0].annotate(f"{count:,}\n({pct:.1f}%)",
                     (bar.get_x() + bar.get_width() / 2., bar.get_height()),
                     ha='center', va='bottom', fontsize=9.5, fontweight='bold', xytext=(0, 4), textcoords='offset points')

# Right: Proportion Across Train / Val / Test Partitions
part_df = pd.DataFrame({
    'Train Set': pd.Series(y_train).value_counts(normalize=True).sort_index() * 100,
    'Val Set': pd.Series(y_val).value_counts(normalize=True).sort_index() * 100,
    'Test Set': pd.Series(y_test).value_counts(normalize=True).sort_index() * 100
})
part_df.index = esi_labels
part_df.plot(kind='bar', ax=axes[1], colormap='viridis', edgecolor='black', alpha=0.9)
axes[1].set_title("Stratified Class Proportion Across Partitions (%)", fontsize=12, fontweight='bold', pad=10)
axes[1].set_xlabel("ESI Acuity Level", fontsize=11, fontweight='bold')
axes[1].set_ylabel("Class Proportion (%)", fontsize=11, fontweight='bold')
axes[1].legend(title="Partition", fontsize=9.5)
axes[1].grid(True, linestyle='--', alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
target_dist_path = os.path.join(plots_dir, "fedmml_target_class_distribution.png")
plt.savefig(target_dist_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_target_class_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Target Class Distribution plot saved to: {target_dist_path}")

# 2. Plot Feature Distributions Stratified by ESI Acuity Level
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()

feature_titles = [
    ('age', 'Age Distribution by ESI Level', 'Age (years)'),
    ('sex_encoded', 'Sex Proportion (1=Male, 0=Female) by ESI Level', 'Sex (0=F, 1=M)'),
    ('systolic_bp', 'Systolic Blood Pressure (SBP) by ESI Level', 'Systolic BP (mmHg)'),
    ('heart_rate', 'Heart Rate (HR) by ESI Level', 'Heart Rate (bpm)'),
    ('respiratory_rate', 'Respiratory Rate (RR) by ESI Level', 'Respiratory Rate (breaths/min)'),
    ('spo2', 'Oxygen Saturation (SpO2) by ESI Level', 'SpO2 (%)')
]

for i, (col, title, xlabel) in enumerate(feature_titles):
    ax = axes[i]
    if col == 'sex_encoded':
        sex_by_esi = df.groupby(target_col)[col].value_counts(normalize=True).unstack() * 100
        sex_by_esi.index = esi_labels
        sex_by_esi.columns = ['Female', 'Male']
        sex_by_esi.plot(kind='bar', stacked=True, ax=ax, color=['#e377c2', '#1f77b4'], edgecolor='black', alpha=0.85)
        ax.set_title(title, fontsize=11.5, fontweight='bold', pad=8)
        ax.set_xlabel("ESI Level", fontsize=10.5, fontweight='bold')
        ax.set_ylabel("Proportion (%)", fontsize=10.5, fontweight='bold')
        ax.legend(title="Gender", fontsize=9.0)
        ax.tick_params(axis='x', rotation=0)
    else:
        clean_sub = df.dropna(subset=[col, target_col])
        sns.boxplot(data=clean_sub, x=target_col, y=col, ax=ax, palette=esi_colors, showfliers=False, width=0.55)
        ax.set_xticklabels(esi_labels)
        ax.set_title(title, fontsize=11.5, fontweight='bold', pad=8)
        ax.set_xlabel("ESI Level", fontsize=10.5, fontweight='bold')
        ax.set_ylabel(xlabel, fontsize=10.5, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.3, axis='y')

plt.tight_layout()
feat_dist_path = os.path.join(plots_dir, "fedmml_feature_distributions.png")
plt.savefig(feat_dist_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_feature_distributions.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Feature Distributions plot saved to: {feat_dist_path}")

In [ ]:
# ---------------------------------------------------------
# Step 3: MICE Imputation (IterativeImputer) & Feature Standardization
# ---------------------------------------------------------
print("=" * 85)
print("  MULTIVARIATE IMPUTATION BY CHAINED EQUATIONS (MICE)")
print("  (Fit exclusively on Training set to prevent validation/test leakage)")
print("=" * 85)

t0 = time.time()
mice_imputer = IterativeImputer(max_iter=10, random_state=42, verbose=0)

# 1. Fit MICE on training encounters and transform all sets
X_train_imp = mice_imputer.fit_transform(X_train_raw)
X_val_imp   = mice_imputer.transform(X_val_raw)
X_test_imp  = mice_imputer.transform(X_test_raw)

print(f"✓ MICE Imputation completed in {time.time()-t0:.2f}s!")
print(f"  - Train null count: {np.isnan(X_train_imp).sum()}")
print(f"  - Val null count  : {np.isnan(X_val_imp).sum()}")
print(f"  - Test null count : {np.isnan(X_test_imp).sum()}")
print("-" * 85)

# 2. Standardize Continuous Features (age, systolic_bp, heart_rate, respiratory_rate, spo2)
cont_indices = [0, 2, 3, 4, 5]

scaler = StandardScaler()
X_train = X_train_imp.copy()
X_val   = X_val_imp.copy()
X_test  = X_test_imp.copy()

X_train[:, cont_indices] = scaler.fit_transform(X_train_imp[:, cont_indices])
X_val[:, cont_indices]   = scaler.transform(X_val_imp[:, cont_indices])
X_test[:, cont_indices]  = scaler.transform(X_test_imp[:, cont_indices])

print(f"✓ Feature Normalization complete: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Vectorized DeLong Method for AUROC Variance & 95% Confidence Intervals
# Reference: DeLong et al. (1988), Biometrics 44(3):837-845
# ---------------------------------------------------------
def delong_roc_variance(ground_truth_binary, predictions_continuous, alpha=0.05):
    """
    Computes exact AUROC, DeLong asymptotic standard error (SE), and (1 - alpha)% confidence interval.
    Vectorized O(N log N) algorithm using sorted structural midranks.
    """
    pos = predictions_continuous[ground_truth_binary == 1]
    neg = predictions_continuous[ground_truth_binary == 0]
    m = len(pos)
    n = len(neg)
    
    if m == 0 or n == 0:
        return 0.0, 0.0, (0.0, 0.0), "[0.0000 - 0.0000]"
    
    pos_sorted = np.sort(pos)
    neg_sorted = np.sort(neg)
    
    # Structural components V10 and V01 via fast binary search
    v10 = (np.searchsorted(neg_sorted, pos, side='left') + np.searchsorted(neg_sorted, pos, side='right')) / (2.0 * n)
    v01 = 1.0 - (np.searchsorted(pos_sorted, neg, side='left') + np.searchsorted(pos_sorted, neg, side='right')) / (2.0 * m)
    
    auc = float(np.mean(v10))
    s10 = float(np.var(v10, ddof=1)) if m > 1 else 0.0
    s01 = float(np.var(v01, ddof=1)) if n > 1 else 0.0
    
    variance = (s10 / m) + (s01 / n)
    se = float(np.sqrt(max(0.0, variance)))
    
    z_crit = 1.959963984540054  # 95% Normal critical value
    ci_lower = max(0.0, auc - z_crit * se)
    ci_upper = min(1.0, auc + z_crit * se)
    ci_str = f"[{ci_lower:.4f} - {ci_upper:.4f}]"
    
    return auc, se, (ci_lower, ci_upper), ci_str

print("✓ Vectorized DeLong's Method for AUROC Variance & 95% CI loaded successfully!")

In [ ]:
# ---------------------------------------------------------
# Step 5: Optuna Hyperparameter Tuning for LightGBM on Validation Set
# ---------------------------------------------------------
print("=" * 85)
print("  OPTUNA HYPERPARAMETER TUNING: LIGHTGBM CLASSIFIER (ON VALIDATION SET)")
print("=" * 85)

def objective_lgbm(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 5,
        'metric': 'multi_logloss',
        'class_weight': 'balanced',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_estimators': 250,
        'verbosity': -1,
        'random_state': 42
    }
    
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train - 1,
        eval_set=[(X_val, y_val - 1)],
        callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
    )
    
    val_preds = model.predict(X_val) + 1
    val_bal_acc = balanced_accuracy_score(y_val, val_preds)
    return val_bal_acc

t_opt_lgb = time.time()
study_lgbm = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, n_trials=30, timeout=180, show_progress_bar=False)

print(f"✓ LightGBM Tuning Complete in {time.time()-t_opt_lgb:.1f}s across {len(study_lgbm.trials)} trials!")
print(f"  * Best Trial Number           : #{study_lgbm.best_trial.number}")
print(f"  * Best Validation Balanced Acc : {study_lgbm.best_value*100:.2f}%")
print("\nOptimal LightGBM Hyperparameters:")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, float):
        print(f"  - {k:20s}: {v:.6f}")
    else:
        print(f"  - {k:20s}: {v}")
print("-" * 85)

# Fit Final Tuned Base LightGBM Model
best_lgb_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'class_weight': 'balanced',
    'n_estimators': 300,
    'verbosity': -1,
    'random_state': 42,
    **study_lgbm.best_params
}
best_lgbm_model = lgb.LGBMClassifier(**best_lgb_params)
best_lgbm_model.fit(
    X_train, y_train - 1,
    eval_set=[(X_val, y_val - 1)],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
)
print(f"✓ Final Tuned Base LightGBM model fitted (Best Iteration: {best_lgbm_model.best_iteration_})!")

In [ ]:
# ---------------------------------------------------------
# Step 6: Train Stage-2 Conditional ESI 4 & 5 Logistic Calibrator (Approach 2)
# ---------------------------------------------------------
print("=" * 85)
print("  TRAINING CONDITIONAL ESI 4 vs. 5 SPECIALIST LOGISTIC CALIBRATOR (STAGE 2)")
print("=" * 85)

# Predict validation probabilities with base LightGBM
val_probs_base = best_lgbm_model.predict_proba(X_val)

# Extract low-acuity validation encounters (ESI 4 & 5)
mask_val_45 = np.isin(y_val, [4, 5])
X_val_calib_inputs = np.hstack([X_val[mask_val_45], val_probs_base[mask_val_45, 3:5]])
y_val_calib_target = (y_val[mask_val_45] == 5).astype(int)  # 1 = ESI 5, 0 = ESI 4

calibrator_45 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
calibrator_45.fit(X_val_calib_inputs, y_val_calib_target)

print(f"✓ Conditional ESI 4/5 Logistic Calibrator trained on {np.sum(mask_val_45):,} low-acuity validation encounters!")
print(f"  - Calibrator Intercept: {calibrator_45.intercept_[0]:.4f}")
print(f"  - Calibrator Coefficients: {np.round(calibrator_45.coef_[0], 4)}")
print("=" * 85)

In [ ]:
# ---------------------------------------------------------
# Step 7: Holdout Test Evaluation & Direct Comparison (Base vs Calibrated LightGBM)
# ---------------------------------------------------------
def compute_comprehensive_metrics(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, precs, f1s, aucs, ses = [], [], [], [], [], [], []
    
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1   = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        
        auc, se, ci, ci_str = delong_roc_variance(y_bin_true, probs[:, idx])
        
        recalls.append(rec); specs.append(spec); bal_accs.append(bal)
        precs.append(prec); f1s.append(f1); aucs.append(auc); ses.append(se)
        
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall_Sensitivity': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'Precision_PPV': round(prec, 4),
            'F1_Score': round(f1, 4),
            'AUROC': round(auc, 4),
            'DeLong_SE': round(se, 4),
            'DeLong_95_CI': ci_str
        })
    
    macro_auc = float(np.mean(aucs))
    macro_se  = float(np.sqrt(np.sum(np.array(ses)**2)) / len(classes))
    macro_ci_low = max(0.0, macro_auc - 1.96 * macro_se)
    macro_ci_up  = min(1.0, macro_auc + 1.96 * macro_se)
    macro_ci_str = f"[{macro_ci_low:.4f} - {macro_ci_up:.4f}]"
    
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall_Sensitivity': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'Precision_PPV': round(np.mean(precs), 4),
        'F1_Score': round(np.mean(f1s), 4),
        'AUROC': round(macro_auc, 4),
        'DeLong_SE': round(macro_se, 4),
        'DeLong_95_CI': macro_ci_str
    })
    return pd.DataFrame(rows)

# 1. Model 1 (Base LightGBM) Predictions on Test Set
probs_test_base = best_lgbm_model.predict_proba(X_test)
preds_test_base = np.argmax(probs_test_base, axis=1) + 1
report_base     = compute_comprehensive_metrics(y_test, preds_test_base, probs_test_base, 'Base_Tuned_LightGBM')

# 2. Model 2 (Conditional Calibrated LightGBM) Predictions on Test Set
X_test_calib_inputs = np.hstack([X_test, probs_test_base[:, 3:5]])
q5_test = calibrator_45.predict_proba(X_test_calib_inputs)[:, 1]
q4_test = 1.0 - q5_test

probs_test_calib = probs_test_base.copy()
p45_sum_test = probs_test_base[:, 3] + probs_test_base[:, 4]
probs_test_calib[:, 3] = p45_sum_test * q4_test
probs_test_calib[:, 4] = p45_sum_test * q5_test
preds_test_calib = np.argmax(probs_test_calib, axis=1) + 1

report_calib = compute_comprehensive_metrics(y_test, preds_test_calib, probs_test_calib, 'Calibrated_LightGBM')

# 3. Side-by-Side Comparison Table
comp_rows = []
for i in range(len(report_base)):
    cls_lbl = report_base.loc[i, 'Class']
    
    rec_base  = report_base.loc[i, 'Recall_Sensitivity']
    rec_calib = report_calib.loc[i, 'Recall_Sensitivity']
    
    bal_base  = report_base.loc[i, 'Balanced_Accuracy']
    bal_calib = report_calib.loc[i, 'Balanced_Accuracy']
    
    auc_base  = report_base.loc[i, 'AUROC']
    auc_calib = report_calib.loc[i, 'AUROC']
    
    f1_base   = report_base.loc[i, 'F1_Score']
    f1_calib  = report_calib.loc[i, 'F1_Score']
    
    comp_rows.append({
        'Class': cls_lbl,
        'Base_Recall': f"{rec_base*100:.2f}%",
        'Calibrated_Recall': f"{rec_calib*100:.2f}%",
        'Delta_Recall': f"{(rec_calib - rec_base)*100:+.2f}%",
        'Base_BalAcc': f"{bal_base*100:.2f}%",
        'Calibrated_BalAcc': f"{bal_calib*100:.2f}%",
        'Delta_BalAcc': f"{(bal_calib - bal_base)*100:+.2f}%",
        'Base_F1': f"{f1_base:.4f}",
        'Calibrated_F1': f"{f1_calib:.4f}",
        'Delta_F1': f"{(f1_calib - f1_base):+.4f}",
        'Base_AUROC': f"{auc_base:.4f}",
        'Calibrated_AUROC': f"{auc_calib:.4f}"
    })

comp_df = pd.DataFrame(comp_rows)

print("=" * 135)
print("   HOLDOUT TEST SET COMPARISON: BASE LIGHTGBM vs CONDITIONAL CALIBRATED LIGHTGBM (N = {:,})".format(len(y_test)))
print("=" * 135)
print(comp_df.to_string(index=False))
print("=" * 135 + chr(10))

# Export CSV Reports
reports_dir = f"{ROOT}/reports"
os.makedirs(reports_dir, exist_ok=True)

report_base.to_csv(os.path.join(reports_dir, 'fedmml_base_lightgbm_report.csv'), index=False)
report_calib.to_csv(os.path.join(reports_dir, 'fedmml_calibrated_lightgbm_report.csv'), index=False)
comp_df.to_csv(os.path.join(reports_dir, 'fedmml_base_vs_calibrated_comparison.csv'), index=False)
print(f"✓ Detailed comparative performance reports saved to {reports_dir}/")

In [ ]:
# ---------------------------------------------------------
# Step 8: Diagnostic Visualizations (1x2 Confusion Matrix, 1x2 ROC Curves & Feature Importance)
# ---------------------------------------------------------
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# 1. 1x2 Side-by-Side Normalized Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

def render_cm(ax, y_true, y_pred, title_text, cmap='Blues'):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    annot = np.empty_like(cm, dtype=object)
    for i in range(5):
        for j in range(5):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap=cmap, cbar=True, ax=ax,
                vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels)
    ax.set_title(title_text, fontsize=11.5, fontweight='bold', pad=10)
    ax.set_xlabel("Predicted ESI Level", fontsize=10.5, fontweight='bold')
    ax.set_ylabel("True ESI Level", fontsize=10.5, fontweight='bold')

render_cm(axes[0], y_test, preds_test_base,
          f"[Base Tuned LightGBM] Normalized Confusion Matrix\nMacro Bal Acc: {report_base.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {report_base.loc[5, 'AUROC']:.4f}", cmap='Blues')

render_cm(axes[1], y_test, preds_test_calib,
          f"[Calibrated LightGBM (Stage-2 ESI 4/5)] Confusion Matrix\nMacro Bal Acc: {report_calib.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {report_calib.loc[5, 'AUROC']:.4f}", cmap='Purples')

plt.tight_layout()
cm_comp_path = os.path.join(plots_dir, "fedmml_base_vs_calibrated_confusion_matrix.png")
plt.savefig(cm_comp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_base_vs_calibrated_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Confusion Matrix comparison saved to: {cm_comp_path}")

# 2. 1x2 Multiclass ROC-AUC Curves with DeLong 95% Confidence Intervals
classes = [1, 2, 3, 4, 5]
n_classes = len(classes)
y_test_bin = label_binarize(y_test, classes=classes)
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

fig, axes = plt.subplots(1, 2, figsize=(19, 8))

def plot_roc_on_ax(ax, y_bin, probs, report_df, model_name):
    fpr, tpr, roc_aucs = dict(), dict(), dict()
    for i, cls in enumerate(classes):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_aucs[i] = auc(fpr[i], tpr[i])
    
    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), probs.ravel())
    roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])
    
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])
    
    macro_ci = report_df.loc[5, 'DeLong_95_CI']
    macro_val = report_df.loc[5, 'AUROC']
    
    ax.plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
    ax.plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUC = {macro_val:.4f}, 95% CI {macro_ci})", color='#17becf', linestyle='--', linewidth=2.5)
    
    for i in range(5):
        cls_ci = report_df.loc[i, 'DeLong_95_CI']
        ax.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUC = {report_df.loc[i, 'AUROC']:.4f}, 95% CI {cls_ci})")
    
    ax.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
    ax.set_title(f"{model_name} ROC-AUC Curves\n(With DeLong 95% Confidence Intervals)", fontsize=11.5, fontweight='bold', pad=10)
    ax.legend(loc="lower right", fontsize=9.0, frameon=True, framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.4)

plot_roc_on_ax(axes[0], y_test_bin, probs_test_base, report_base, "[Model 1: Base LightGBM]")
plot_roc_on_ax(axes[1], y_test_bin, probs_test_calib, report_calib, "[Model 2: Calibrated LightGBM]")

plt.tight_layout()
roc_comp_path = os.path.join(plots_dir, "fedmml_base_vs_calibrated_roc_auc_curve.png")
plt.savefig(roc_comp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_base_vs_calibrated_roc_auc_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ ROC-AUC comparison curves saved to: {roc_comp_path}")

# 3. Feature Importance Visualization
fig, ax = plt.subplots(figsize=(9, 5))
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance_Gain': best_lgbm_model.booster_.feature_importance(importance_type='gain'),
    'Importance_Split': best_lgbm_model.booster_.feature_importance(importance_type='split')
}).sort_values('Importance_Gain', ascending=False)

sns.barplot(data=feat_imp, x='Importance_Gain', y='Feature', palette='Blues_r', ax=ax, edgecolor='black')
ax.set_title('Tuned LightGBM Feature Importance (Information Gain on FedMML Features)', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Total Information Gain', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature Name', fontsize=11, fontweight='bold')

for p in ax.patches:
    ax.annotate(f"{p.get_width():,.1f}", (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=9.5, fontweight='bold', xytext=(5, 0), textcoords='offset points')

plt.tight_layout()
feat_imp_path = os.path.join(plots_dir, "fedmml_lightgbm_tuned_feature_importance.png")
plt.savefig(feat_imp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_tuned_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Feature Importance plot saved to: {feat_imp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 9: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f"{ROOT}/deploy"
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'mice_imputer': mice_imputer,
    'scaler': scaler,
    'feature_names': feature_names,
    'model_lgbm': best_lgbm_model,
    'calibrator_45': calibrator_45,
    'best_params_lgbm': study_lgbm.best_params,
    'best_val_score': study_lgbm.best_value,
    'config_training': config['training']
}

bundle_file = os.path.join(deploy_dir, 'fedmml_calibrated_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='FedMML_Optuna_Tuned_LightGBM_With_Conditional_ESI45_Logistic_Calibrator',
    dataset='datasets/fedmml_ed_triage_dataset.csv',
    config_file='config/triage_conf.json',
    config_training=config['training'],
    imputation_method='MICE_IterativeImputer (fit on Train partition)',
    tuning_framework='Optuna (TPESampler)',
    tuning_objective='Validation Macro Balanced Accuracy',
    best_hyperparameters=study_lgbm.best_params,
    best_validation_balanced_accuracy=study_lgbm.best_value,
    calibrator_type='Stage-2 Binary Logistic Regressor on ESI 4 vs 5 (class_weight=balanced)',
    auroc_confidence_intervals='DeLong non-parametric U-statistic method (95% Wald CI)',
    total_encounters=len(df),
    train_encounters=len(X_train),
    val_encounters=len(X_val),
    test_encounters=len(X_test),
    features=['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2'],
    target='esi_level',
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    comparison_summary=comp_rows
)

manifest_file = os.path.join(deploy_dir, 'fedmml_calibrated_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Production Bundle   : {bundle_file}")
print(f"✓ Production Manifest : {manifest_file}")